# Data Processing

Build synthetic/interpolated datasets from raw IPR data. Raw files are never modified; processed outputs are regenerated under `Data/Processed`.

Log/interpolation workflow:

1. Start from raw IPR values.
2. Compute `log10(raw IPR)`.
3. Fit `make_smoothing_spline` to `log10(IPR)` on the raw grid.
4. Evaluate the smoothed log curve on the synthetic grid.
5. Save the interpolated log values as `log10_*.csv`.
6. Exponentiate back with `10**log10(IPR)`.
7. Save the raw interpolated values as the processed IPR CSV files.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from scipy.interpolate import make_smoothing_spline

DATA_DIR = Path.cwd()
if DATA_DIR.name != 'Data':
    DATA_DIR = Path('Data')

RAW_DIR = DATA_DIR / 'Raw'
PROCESSED_DIR = DATA_DIR / 'Processed'

RAW_TFIM_1D_DIR = RAW_DIR / 'Data TFIM' / 'Data TFIM 1D'
RAW_TFIM_2D_DIR = RAW_DIR / 'Data TFIM' / 'Data TFIM 2D'
RAW_HEISENBERG_DIR = RAW_DIR / 'Data Heisenberg'

DATASYN_TFIM_1D_DIR = PROCESSED_DIR / 'DataSyn TFIM' / 'DataSyn TFIM 1D'
DATASYN_TFIM_2D_DIR = PROCESSED_DIR / 'DataSyn TFIM' / 'DataSyn TFIM 2D'
DATASYN_HEISENBERG_DIR = PROCESSED_DIR / 'DataSyn Heisenberg'

TFIM_SYN_POINTS = 4000       # h in [0, 4]
HEISENBERG_SYN_POINTS = 5000 # Delta in [0, 5]

# None lets scipy choose the smoothing strength by generalized cross-validation.
IPR_SMOOTHING_LAMBDA = None


## Generic Smoothing-Spline Helpers


In [2]:
def _prepare_unique_xy(x: np.ndarray, y: np.ndarray):
    """Sort x and average y on duplicate x values for stable interpolation."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    order = np.argsort(x)
    x_sorted = x[order]
    y_sorted = y[order]

    x_unique, inv = np.unique(x_sorted, return_inverse=True)
    if x_unique.size == x_sorted.size:
        return x_unique, y_sorted

    y_sum = np.zeros_like(x_unique, dtype=float)
    np.add.at(y_sum, inv, y_sorted)
    counts = np.bincount(inv)
    return x_unique, y_sum / counts


def smoothing_spline_interpolate_series(
    x: np.ndarray,
    y: np.ndarray,
    x_new: np.ndarray,
    smoothing_lambda: float | None = IPR_SMOOTHING_LAMBDA,
):
    """Evaluate a smoothing spline on x_new."""
    x_u, y_u = _prepare_unique_xy(x, y)
    if x_u.size < 2:
        fill = y_u[0] if x_u.size == 1 else np.nan
        return np.full_like(np.asarray(x_new, dtype=float), fill, dtype=float)

    spline = make_smoothing_spline(x_u, y_u, lam=smoothing_lambda)
    x_new = np.asarray(x_new, dtype=float)
    x_clip = np.clip(x_new, x_u[0], x_u[-1])
    return np.asarray(spline(x_clip), dtype=float)


def interpolate_ipr_columns(
    grid: np.ndarray,
    columns: dict[str, np.ndarray],
    n_points: int,
    grid_min: float | None = None,
    grid_max: float | None = None,
    smoothing_lambda: float | None = IPR_SMOOTHING_LAMBDA,
):
    """Interpolate IPR columns on a uniform grid in log10(IPR) space."""
    grid = np.asarray(grid, dtype=float)
    grid_min = float(np.min(grid)) if grid_min is None else float(grid_min)
    grid_max = float(np.max(grid)) if grid_max is None else float(grid_max)
    grid_syn = np.linspace(grid_min, grid_max, int(n_points))

    raw_syn = {}
    log_syn = {}
    for name, values in columns.items():
        values = np.asarray(values, dtype=float)
        if values.shape != grid.shape:
            raise ValueError(f'{name}: shape {values.shape} does not match grid shape {grid.shape}.')

        if np.any(values <= 0.0):
            raise ValueError(f'{name}: non-positive IPR value found; cannot interpolate log10(IPR).')
        log_values = np.log10(values)
        log_interp = smoothing_spline_interpolate_series(grid, log_values, grid_syn, smoothing_lambda=smoothing_lambda)
        raw_syn[name] = np.power(10.0, log_interp)
        log_syn[f'log10_{name}'] = log_interp

    return grid_syn, raw_syn, log_syn


def read_vector_csv(path: Path, column: str | None = None):
    """Read either a plain one-column numeric file or a CSV with a named column."""
    path = Path(path)
    try:
        return np.loadtxt(path, delimiter=',', dtype=float)
    except ValueError:
        df = pd.read_csv(path)
        if column is None:
            if df.shape[1] != 1:
                raise ValueError(f'{path} has multiple columns; pass column explicitly.')
            column = df.columns[0]
        return df[column].to_numpy(dtype=float)


def write_plain_vector(path: Path, values: np.ndarray):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savetxt(path, np.asarray(values, dtype=float), delimiter=',')


def write_value_csv(path: Path, values: np.ndarray):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({'value': np.asarray(values, dtype=float)}).to_csv(path, index=False)


## TFIM 1D Synthetic Data

Input: `Data/Raw/Data TFIM/Data TFIM 1D/L=<N>x1/{hs,ipr_x,ipr_z}.csv`

Output: `Data/Processed/DataSyn TFIM/DataSyn TFIM 1D/L=<N>x1/` with 4000 points on `h in [0, 4]`.

In [3]:
def process_tfim_1d_dataset(
    raw_dir: Path = RAW_TFIM_1D_DIR,
    out_dir: Path = DATASYN_TFIM_1D_DIR,
    n_points: int = TFIM_SYN_POINTS,
):
    raw_dir = Path(raw_dir)
    out_dir = Path(out_dir)
    folders = sorted(
        [p for p in raw_dir.iterdir() if p.is_dir() and re.match(r'L=\d+x1$', p.name)],
        key=lambda p: int(p.name.removeprefix('L=').removesuffix('x1')),
    )

    summary = []
    for folder in folders:
        h = read_vector_csv(folder / 'hs.csv')
        ipr_x = read_vector_csv(folder / 'ipr_x.csv')
        ipr_z = read_vector_csv(folder / 'ipr_z.csv')

        h_syn, raw_syn, log_syn = interpolate_ipr_columns(
            h,
            {'ipr_x': ipr_x, 'ipr_z': ipr_z},
            n_points=n_points,
            grid_min=0.0,
            grid_max=4.0,
        )

        dest = out_dir / folder.name
        write_plain_vector(dest / 'hs.csv', h_syn)
        write_plain_vector(dest / 'ipr_x.csv', raw_syn['ipr_x'])
        write_plain_vector(dest / 'ipr_z.csv', raw_syn['ipr_z'])
        write_plain_vector(dest / 'log10_ipr_x.csv', log_syn['log10_ipr_x'])
        write_plain_vector(dest / 'log10_ipr_z.csv', log_syn['log10_ipr_z'])

        summary.append({'folder': folder.name, 'points': len(h_syn), 'h_min': h_syn[0], 'h_max': h_syn[-1]})

    return pd.DataFrame(summary)


tfim_summary = process_tfim_1d_dataset()
tfim_summary.head(), tfim_summary.tail(), len(tfim_summary)


(  folder  points  h_min  h_max
 0  L=3x1    4000    0.0    4.0
 1  L=4x1    4000    0.0    4.0
 2  L=5x1    4000    0.0    4.0
 3  L=6x1    4000    0.0    4.0
 4  L=7x1    4000    0.0    4.0,
      folder  points  h_min  h_max
 93   L=96x1    4000    0.0    4.0
 94   L=97x1    4000    0.0    4.0
 95   L=98x1    4000    0.0    4.0
 96   L=99x1    4000    0.0    4.0
 97  L=100x1    4000    0.0    4.0,
 98)

## TFIM 2D Synthetic Data

Input: `Data/Raw/Data TFIM/Data TFIM 2D/L=<N>x<N>/{hs,ipr_x,ipr_z}.csv`

Output: `Data/Processed/DataSyn TFIM/DataSyn TFIM 2D/L=<N>x<N>/` with 4000 points on `h in [0, 4]`.

In [4]:
def process_tfim_2d_dataset(
    raw_dir: Path = RAW_TFIM_2D_DIR,
    out_dir: Path = DATASYN_TFIM_2D_DIR,
    n_points: int = TFIM_SYN_POINTS,
):
    raw_dir = Path(raw_dir)
    out_dir = Path(out_dir)
    folders = sorted(
        [p for p in raw_dir.iterdir() if p.is_dir() and re.match(r'L=\d+x\d+$', p.name)],
        key=lambda p: int(p.name.removeprefix('L=').split('x')[0]),
    )

    summary = []
    for folder in folders:
        h = read_vector_csv(folder / 'hs.csv')
        ipr_x = read_vector_csv(folder / 'ipr_x.csv')
        ipr_z = read_vector_csv(folder / 'ipr_z.csv')

        h_syn, raw_syn, log_syn = interpolate_ipr_columns(
            h,
            {'ipr_x': ipr_x, 'ipr_z': ipr_z},
            n_points=n_points,
            grid_min=0.0,
            grid_max=4.0,
        )

        dest = out_dir / folder.name
        write_plain_vector(dest / 'hs.csv', h_syn)
        write_plain_vector(dest / 'ipr_x.csv', raw_syn['ipr_x'])
        write_plain_vector(dest / 'ipr_z.csv', raw_syn['ipr_z'])
        write_plain_vector(dest / 'log10_ipr_x.csv', log_syn['log10_ipr_x'])
        write_plain_vector(dest / 'log10_ipr_z.csv', log_syn['log10_ipr_z'])

        summary.append({'folder': folder.name, 'points': len(h_syn), 'h_min': h_syn[0], 'h_max': h_syn[-1]})

    return pd.DataFrame(summary)


tfim_2d_summary = process_tfim_2d_dataset()
tfim_2d_summary.head(), tfim_2d_summary.tail(), len(tfim_2d_summary)


(  folder  points  h_min  h_max
 0  L=3x3    4000    0.0    4.0
 1  L=4x4    4000    0.0    4.0
 2  L=5x5    4000    0.0    4.0
 3  L=6x6    4000    0.0    4.0
 4  L=7x7    4000    0.0    4.0,
     folder  points  h_min  h_max
 3    L=6x6    4000    0.0    4.0
 4    L=7x7    4000    0.0    4.0
 5    L=8x8    4000    0.0    4.0
 6    L=9x9    4000    0.0    4.0
 7  L=10x10    4000    0.0    4.0,
 8)

## Heisenberg Synthetic Data

Input: `Data/Raw/Data Heisenberg/L=<N>/{IPR X,IPR XY,IPR Z}.csv`

Output: `Data/Processed/DataSyn Heisenberg/L=<N>/` with 5000 points on `Delta in [0, 5]`.

In [5]:
def process_heisenberg_dataset(
    raw_dir: Path = RAW_HEISENBERG_DIR,
    out_dir: Path = DATASYN_HEISENBERG_DIR,
    n_points: int = HEISENBERG_SYN_POINTS,
):
    raw_dir = Path(raw_dir)
    out_dir = Path(out_dir)
    folders = sorted(
        [p for p in raw_dir.iterdir() if p.is_dir() and re.match(r'L=\d+$', p.name)],
        key=lambda p: int(p.name.removeprefix('L=')),
    )

    source_delta = np.linspace(0.0, 5.0, 51)
    source_files = ['IPR X.csv', 'IPR XY.csv', 'IPR Z.csv']

    summary = []
    for folder in folders:
        columns = {}
        for file_name in source_files:
            key = file_name.removesuffix('.csv').replace(' ', '_')
            values = read_vector_csv(folder / file_name, column='value')
            if values.size != source_delta.size:
                raise ValueError(f'{folder.name}/{file_name}: expected 51 values, got {values.size}.')
            columns[key] = values

        delta_syn, raw_syn, log_syn = interpolate_ipr_columns(
            source_delta,
            columns,
            n_points=n_points,
            grid_min=0.0,
            grid_max=5.0,
        )

        dest = out_dir / folder.name
        write_plain_vector(dest / 'Delta.csv', delta_syn)
        for file_name in source_files:
            key = file_name.removesuffix('.csv').replace(' ', '_')
            write_value_csv(dest / file_name, raw_syn[key])
            write_value_csv(dest / f'log10_{key}.csv', log_syn[f'log10_{key}'])

        summary.append({'folder': folder.name, 'points': len(delta_syn), 'Delta_min': delta_syn[0], 'Delta_max': delta_syn[-1]})

    return pd.DataFrame(summary)


heisenberg_summary = process_heisenberg_dataset()
heisenberg_summary.head(), heisenberg_summary.tail(), len(heisenberg_summary)


(  folder  points  Delta_min  Delta_max
 0    L=6    5000        0.0        5.0
 1    L=8    5000        0.0        5.0
 2   L=10    5000        0.0        5.0
 3   L=12    5000        0.0        5.0
 4   L=14    5000        0.0        5.0,
   folder  points  Delta_min  Delta_max
 5   L=16    5000        0.0        5.0
 6   L=18    5000        0.0        5.0
 7   L=20    5000        0.0        5.0
 8   L=22    5000        0.0        5.0
 9   L=24    5000        0.0        5.0,
 10)

## Thermal TFIM SVM Dataset

Build the processed thermal subset used for dense SVM tests. Gram matrix plots use raw thermal data. The processed subset uses `L = 20, 50, 100`, `beta = 0.1, ..., 2.0`, and 4000 dense field values on `h in [0, 4]`.

IPR data are stored as CSV feature tables. HS data are stored as dense `float32` `.npy` Gram matrices because each processed HS matrix has shape `4000 x 4000`.

HS interpolation process:

- For each `L` and `beta`, extract the raw HS Gram matrix from `HS Kernel L=<L>.csv`.
- Treat the matrix as a 2D function `K(h_a, h_b)` on the original field grid.
- Use `RegularGridInterpolator` to evaluate `K(h_a, h_b)` on the dense `h_dense x h_dense` grid.
- Normalize the dense interpolated matrix with `K_ij / sqrt(K_ii K_jj)` so the processed diagonal is fixed back to 1.
- Symmetrize the dense matrix with `(K + K.T) / 2` to remove interpolation or numerical asymmetries.
- Store each dense matrix as `float32` in a separate `.npy` file, one file per `(L, beta)` pair.

In [6]:
from scipy.interpolate import RegularGridInterpolator

RAW_THERMAL_DIR = RAW_DIR / 'Data TFIM Thermal'
RAW_THERMAL_IPR_DIR = RAW_THERMAL_DIR / 'Data IPR Kernel'
RAW_THERMAL_HS_DIR = RAW_THERMAL_DIR / 'Data HS Kernel'

DATASYN_TFIM_THERMAL_DIR = PROCESSED_DIR / 'DataSyn TFIM Thermal'
DATASYN_THERMAL_IPR_DIR = DATASYN_TFIM_THERMAL_DIR / 'DataSyn IPR Kernel'
DATASYN_THERMAL_HS_DIR = DATASYN_TFIM_THERMAL_DIR / 'DataSyn HS Kernel'
DATASYN_THERMAL_GRID_PATH = DATASYN_TFIM_THERMAL_DIR / 'h_dense_4000.csv'

THERMAL_L_VALUES = [20, 50, 100]
THERMAL_BETA_VALUES = [float(v) for v in np.round(np.arange(0.1, 2.0 + 1e-12, 0.1), 1)]
THERMAL_SYN_POINTS = 4000
THERMAL_H_DENSE = np.linspace(0.0, 4.0, THERMAL_SYN_POINTS)

# Make sure to close any notebooks/kernels using thermal HS .npy files before running, as they cannot be overwritten while open.
# HS output for this subset is expected to be roughly 4 GB as float32 .npy files.

def normalize_gram_matrix(K: np.ndarray) -> np.ndarray:
    K = np.asarray(K, dtype=float)
    diag = np.diag(K)
    denom = np.sqrt(np.outer(diag, diag))
    return np.divide(K, denom, out=np.zeros_like(K), where=denom > 0)

def beta_file_token(beta: float) -> str:
    return f'{float(beta):.1f}'

def save_npy_atomic(path: Path, array: np.ndarray):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.stem + '.tmp.npy')
    np.save(tmp_path, array)
    try:
        tmp_path.replace(path)
    except OSError as exc:
        try:
            tmp_path.unlink(missing_ok=True)
        except OSError:
            pass
        raise OSError(
            f'Could not replace {path}. On Windows this usually means the existing .npy file '
            f'is still open in another notebook/kernel or memory map. Close/restart notebooks using '
            f'thermal HS kernels, then rerun this cell.'
        ) from exc

def process_thermal_ipr_dataset(
    L_values=THERMAL_L_VALUES,
    beta_values=THERMAL_BETA_VALUES,
    h_dense=THERMAL_H_DENSE,
    raw_dir: Path = RAW_THERMAL_IPR_DIR,
    out_dir: Path = DATASYN_THERMAL_IPR_DIR,
):
    raw_dir = Path(raw_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    write_plain_vector(DATASYN_THERMAL_GRID_PATH, h_dense)

    summary = []
    for L in L_values:
        source = raw_dir / f'ipr_thermal_L={int(L)}.csv'
        df = pd.read_csv(source)
        rows = []
        for beta in beta_values:
            sub = df[np.isclose(df['beta'], float(beta))].sort_values('h')
            if sub.empty:
                raise ValueError(f'{source.name}: beta={beta} not found')

            _, raw_syn, log_syn = interpolate_ipr_columns(
                sub['h'].to_numpy(dtype=float),
                {
                    'ipr_z': sub['ipr_z'].to_numpy(dtype=float),
                    'ipr_x': sub['ipr_x'].to_numpy(dtype=float),
                },
                n_points=len(h_dense),
                grid_min=float(h_dense[0]),
                grid_max=float(h_dense[-1]),
            )

            rows.append(pd.DataFrame({
                'h': h_dense,
                'beta': float(beta),
                'ipr_z': raw_syn['ipr_z'],
                'ipr_x': raw_syn['ipr_x'],
                'log10_ipr_z': log_syn['log10_ipr_z'],
                'log10_ipr_x': log_syn['log10_ipr_x'],
            }))

        out_path = out_dir / f'ipr_thermal_L={int(L)}.csv'
        out_df = pd.concat(rows, ignore_index=True)
        out_df.to_csv(out_path, index=False)
        summary.append({'L': int(L), 'betas': len(beta_values), 'rows': len(out_df), 'path': str(out_path)})

    return pd.DataFrame(summary)

def hs_regular_grid_interpolate(h_a: np.ndarray, h_b: np.ndarray, K_raw: np.ndarray, h_dense: np.ndarray) -> np.ndarray:
    interpolator = RegularGridInterpolator(
        (h_a, h_b),
        K_raw,
        method='linear',
        bounds_error=False,
        fill_value=None,
    )
    left_grid, right_grid = np.meshgrid(h_dense, h_dense, indexing='ij')
    points = np.column_stack([left_grid.ravel(), right_grid.ravel()])
    return interpolator(points).reshape(len(h_dense), len(h_dense))

def process_thermal_hs_dataset(
    L_values=THERMAL_L_VALUES,
    beta_values=THERMAL_BETA_VALUES,
    h_dense=THERMAL_H_DENSE,
    raw_dir: Path = RAW_THERMAL_HS_DIR,
    out_dir: Path = DATASYN_THERMAL_HS_DIR,
):
    raw_dir = Path(raw_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    write_plain_vector(DATASYN_THERMAL_GRID_PATH, h_dense)

    summary = []
    for L in L_values:
        source = raw_dir / f'HS Kernel L={int(L)}.csv'
        df = pd.read_csv(source)
        l_dir = out_dir / f'L={int(L)}'
        l_dir.mkdir(parents=True, exist_ok=True)

        for beta in beta_values:
            sub = df[np.isclose(df['beta'], float(beta))].copy()
            if sub.empty:
                raise ValueError(f'{source.name}: beta={beta} not found')

            h_a = np.sort(sub['h_a'].unique().astype(float))
            h_b = np.sort(sub['h_b'].unique().astype(float))
            K_raw = sub.pivot(index='h_a', columns='h_b', values='kernel_real').reindex(index=h_a, columns=h_b).to_numpy(dtype=float)
            if np.any(~np.isfinite(K_raw)):
                raise ValueError(f'{source.name}: non-finite values for beta={beta}')

            K = hs_regular_grid_interpolate(h_a, h_b, K_raw, h_dense)
            K = normalize_gram_matrix(K)
            K = 0.5 * (K + K.T)
            K = K.astype(np.float32, copy=False)

            out_path = l_dir / f'HS_kernel_beta={beta_file_token(beta)}.npy'
            save_npy_atomic(out_path, K)
            summary.append({
                'L': int(L),
                'beta': float(beta),
                'shape': K.shape,
                'dtype': str(K.dtype),
                'path': str(out_path),
            })

    return pd.DataFrame(summary)

thermal_ipr_summary = process_thermal_ipr_dataset()
thermal_hs_summary = process_thermal_hs_dataset()

thermal_ipr_summary, thermal_hs_summary.head(), len(thermal_hs_summary)


(     L  betas   rows                                               path
 0   20     20  80000  c:\Users\cleme\Qiskit\VSCode Files\IPR Kernel ...
 1   50     20  80000  c:\Users\cleme\Qiskit\VSCode Files\IPR Kernel ...
 2  100     20  80000  c:\Users\cleme\Qiskit\VSCode Files\IPR Kernel ...,
     L  beta         shape    dtype  \
 0  20   0.1  (4000, 4000)  float32   
 1  20   0.2  (4000, 4000)  float32   
 2  20   0.3  (4000, 4000)  float32   
 3  20   0.4  (4000, 4000)  float32   
 4  20   0.5  (4000, 4000)  float32   
 
                                                 path  
 0  c:\Users\cleme\Qiskit\VSCode Files\IPR Kernel ...  
 1  c:\Users\cleme\Qiskit\VSCode Files\IPR Kernel ...  
 2  c:\Users\cleme\Qiskit\VSCode Files\IPR Kernel ...  
 3  c:\Users\cleme\Qiskit\VSCode Files\IPR Kernel ...  
 4  c:\Users\cleme\Qiskit\VSCode Files\IPR Kernel ...  ,
 60)

## Sanity Checks

In [7]:
def count_processed_files(root: Path, pattern: str):
    root = Path(root)
    if not root.exists():
        return 0
    return sum(1 for p in root.rglob(pattern) if p.is_file())


print('TFIM 1D folders:', len(tfim_summary), 'CSV files:', count_processed_files(DATASYN_TFIM_1D_DIR, '*.csv'))
print('TFIM 2D folders:', len(tfim_2d_summary), 'CSV files:', count_processed_files(DATASYN_TFIM_2D_DIR, '*.csv'))
print('Heisenberg folders:', len(heisenberg_summary), 'CSV files:', count_processed_files(DATASYN_HEISENBERG_DIR, '*.csv'))
print('Thermal IPR CSV files:', count_processed_files(DATASYN_THERMAL_IPR_DIR, '*.csv'))
print('Thermal HS NPY files:', count_processed_files(DATASYN_THERMAL_HS_DIR, '*.npy'))


TFIM 1D folders: 98 CSV files: 490
TFIM 2D folders: 8 CSV files: 40
Heisenberg folders: 10 CSV files: 70
Thermal IPR CSV files: 3
Thermal HS NPY files: 60
